In [1]:
import os
import cv2 as cv
import numpy as np
import rasterio
from pathlib import Path
from tqdm import tqdm
from image_processor import ImageProcessor, Bands, ThreeBandInputPaths
import glob


In [3]:
# get all input data from the large field dataset
BASE_INPUT_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/")

sub = ["large", "mid", "small"]
label_paths = []
image_paths = []
output_paths = []

for s in sub:
    print(f"Processing {s} field dataset")
    BASE_INPUT_PATH = Path(f"/home/samuel/test/MasterThesis/Orthomosaics/{s}")

    folders = os.listdir(BASE_INPUT_PATH)
    input_paths = []
    for folder in folders:
        folder_path = BASE_INPUT_PATH / folder
        if folder_path.is_dir():
                input_paths.append(folder_path)

    # find sub_folders in the large field dataset
    sub_folders = []
    for folder in input_paths:
        for sub_folder in folder.iterdir():
            if sub_folder.is_dir():
                sub_folders.append(sub_folder)

    # find all label and image files in the sub_folders

    image_count = 0

    for sub_folder in sub_folders:
        output_paths.append(sub_folder / "combined_output")
        label_path = sub_folder / "labels"
        image_path = sub_folder / "processed_output"
        label_paths.append(label_path)
        image_paths.append(image_path)

        print("Checking image path:", image_path)

        image_path = image_path / "image_tiles"
        image_count += len(list(image_path.glob("*.tif")))

    print(f"Found {image_count} images")

print(len(label_paths), len(image_paths), len(output_paths))

PATHS = [image_paths, output_paths]

proc = ImageProcessor(BASE_INPUT_PATH, BASE_INPUT_PATH, "")


Processing large field dataset
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_500x_500y_rotated_45/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_250x_250y_rotated_45/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x250y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated500x250y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x500y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated500x500y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/processed_output
Checki

In [5]:
extension = ".png"

def norm_index_image(image, min, max):
    norm_image = np.clip(image, min, max)
    norm_image = ((norm_image - min) / (max - min) * 255).astype(np.uint8)
    return norm_image

utf8_input_paths = [str(path / "image_tiles_indeces") for path in image_paths]

utf8_output_paths = [str(path / "image_tiles_indeces_utf8") for path in image_paths]

ndvi_min = -1.00
ndvi_max = 1.00


for INPUT_PATH, OUTPUT_PATH in zip(utf8_input_paths, utf8_output_paths):

    INPUT_PATH = os.path.join(INPUT_PATH, "NDVI")
    OUTPUT_PATH = os.path.join(OUTPUT_PATH, "NDVI")

    os.makedirs(OUTPUT_PATH, exist_ok=True)

    if len(glob.glob(os.path.join(OUTPUT_PATH, f"*{extension}"))) > 0:
        print(f"Files already exist in {OUTPUT_PATH} with extension {extension}")
        continue

    for filepath in tqdm(glob.glob(os.path.join(INPUT_PATH, f"*{extension}"))):
        filename = os.path.basename(filepath)
        if "xml" in filename:
            continue
        with rasterio.open(filepath) as src:
            img = src.read(1)
        
        ndvi = img.astype(np.float32)
        # Convert index image to UTF-8 encoded image
        ndvi_clipped = np.clip(ndvi, ndvi_min, ndvi_max)
        ndvi_norm = np.floor(((ndvi_clipped + 1) / 2) * 255).astype(np.uint8)
        output_path = os.path.join(OUTPUT_PATH, filename)
        with rasterio.open(output_path, 'w', driver='GTiff', height=ndvi_norm.shape[0],
                        width=ndvi_norm.shape[1], count=1, dtype=ndvi_norm.dtype) as dst:
            dst.write(ndvi_norm, 1)
    
ndvi_min = -1.00
ndvi_max = 1.00

for INPUT_PATH, OUTPUT_PATH in zip(utf8_input_paths, utf8_output_paths):

    INPUT_PATH = os.path.join(INPUT_PATH, "NGRDI")
    OUTPUT_PATH = os.path.join(OUTPUT_PATH, "NGRDI")

    os.makedirs(OUTPUT_PATH, exist_ok=True)


    if len(glob.glob(os.path.join(OUTPUT_PATH, f"*{extension}"))) > 0:
        print(f"Files already exist in {OUTPUT_PATH} with extension {extension}")
        continue

    for filepath in tqdm(glob.glob(os.path.join(INPUT_PATH, f"*{extension}"))):
        filename = os.path.basename(filepath)
        with rasterio.open(filepath) as src:
            img = src.read(1)
        
        ndvi = img.astype(np.float32)
        ndvi_clipped = np.clip(ndvi, ndvi_min, ndvi_max)
        ndvi_norm = np.floor(((ndvi_clipped + 1) / 2) * 255).astype(np.uint8)

        output_path = os.path.join(OUTPUT_PATH, filename)
        with rasterio.open(output_path, 'w', driver='GTiff', height=ndvi_norm.shape[0],
                        width=ndvi_norm.shape[1], count=1, dtype=ndvi_norm.dtype) as dst:
            dst.write(ndvi_norm, 1)

rvi_min = 0.00
rvi_max = 9.00

for INPUT_PATH, OUTPUT_PATH in zip(utf8_input_paths, utf8_output_paths):

    INPUT_PATH = os.path.join(INPUT_PATH, "RVI")
    OUTPUT_PATH = os.path.join(OUTPUT_PATH, "RVI")

    os.makedirs(OUTPUT_PATH, exist_ok=True)


    if len(glob.glob(os.path.join(OUTPUT_PATH, f"*{extension}"))) > 0:
        print(f"Files already exist in {OUTPUT_PATH} with extension {extension}")
        continue

    for filepath in tqdm(glob.glob(os.path.join(INPUT_PATH, f"*{extension}"))):
        filename = os.path.basename(filepath)
        with rasterio.open(filepath) as src:
            img = src.read(1)
        
        rvi = img.astype(np.float32)
        rvi_norm = norm_index_image(rvi, rvi_min, rvi_max)

        output_path = os.path.join(OUTPUT_PATH, filename)
        with rasterio.open(output_path, 'w', driver='GTiff', height=rvi_norm.shape[0],
                        width=rvi_norm.shape[1], count=1, dtype=rvi_norm.dtype) as dst:
            dst.write(rvi_norm, 1)


        

        


Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_250x_250y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x250y/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated500x250y/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x500y/processed_output/

  0%|          | 0/452 [00:00<?, ?it/s]/home/samuel/test/MasterThesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
100%|██████████| 452/452 [00:10<00:00, 44.35it/s]


Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/rotated/rotated60/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/original/original/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated_rotated/translated_250x_250y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x250y/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated500x250y/processed_output/image_tiles_indeces_ut

100%|██████████| 235/235 [00:04<00:00, 55.46it/s]


Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/rotated/rotated60/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_250x_250y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x250y/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated500x250y/processed_output/image_tiles_in

...

Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/large/rotated/rotated60/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/original/original/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated_rotated/translated_250x_250y_rotated_45/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x250y/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated500x250y/processed_output/image_tiles_indeces_utf8/RV

100%|██████████| 235/235 [00:04<00:00, 49.93it/s]


Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/mid/rotated/rotated60/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_250x_250y_rotated_45/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x250y/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated500x250y/processed_output/image_tiles_indeces

100%|██████████| 136/136 [00:39<00:00,  3.48it/s]

Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/rotated/rotated60/processed_output/image_tiles_indeces_utf8/RVI with extension .png


In [6]:
# "RED_GREEN_NIR" combination (nir, red, green)
ending = "RED_GREEN_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):
    print(f"Processing input directory: {input_dir} to output directory: {output_dir}")
    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_GREEN_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")

            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir /  "extended_green" / img_name_eg,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                percentiles=[]
            )

    else:
        print("RED_GREEN_NIR image creation skipped; output directory already exists.")

Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal_vertical/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal_vertical/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/proce

RED_GREEN_NIR image creation:   0%|          | 0/452 [00:00<?, ?it/s]/home/samuel/test/MasterThesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
RED_GREEN_NIR image creation: 100%|██████████| 452/452 [01:08<00:00,  6.64it/s]


Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/rotated/rotated60/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/rotated/rotated60/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x250y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x250y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x500y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x500y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/transla

RED_GREEN_NIR image creation: 100%|██████████| 235/235 [00:15<00:00, 15.31it/s]


Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/rotated/rotated60/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/rotated/rotated60/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x250y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x250y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x500y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated250x500y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/mid/translated/translated

RED_GREEN_NIR image creation: 100%|██████████| 136/136 [00:14<00:00,  9.12it/s]

Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/small/rotated/rotated60/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/small/rotated/rotated60/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x250y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x250y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x500y/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/small/translated/translated250x500y/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/small/transla

In [7]:
# RED_NIR_NDVI combination (nir, red, ndvi)
ending = "RED_NIR_NDVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_NIR_NDVI image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "nir" / img_name_nir,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NDVI" / img_name_ndvi,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_NIR_NDVI image creation skipped; output directory already exists.")

RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.


RED_NIR_NDVI image creation: 100%|██████████| 452/452 [01:03<00:00,  7.10it/s]


RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.


RED_NIR_NDVI image creation: 100%|██████████| 235/235 [00:11<00:00, 19.84it/s]


RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.


RED_NIR_NDVI image creation: 100%|██████████| 136/136 [00:17<00:00,  7.98it/s]

RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.
RED_NIR_NDVI image creation skipped; output directory already exists.


In [8]:
# RED_REDEDGE_NIR combination (Nir, red edge, red)
ending = "RED_REDEDGE_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_REDEDGE_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_re = img_name.replace(".tif", "_REDEDGE.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "red_edge" / img_name_re,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_REDEDGE_NIR image creation skipped; output directory already exists.")

RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.


RED_REDEDGE_NIR image creation: 100%|██████████| 452/452 [00:26<00:00, 16.85it/s]


RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.


RED_REDEDGE_NIR image creation: 100%|██████████| 235/235 [00:13<00:00, 18.05it/s]


RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.


RED_REDEDGE_NIR image creation: 100%|██████████| 136/136 [00:07<00:00, 17.62it/s]

RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.
RED_REDEDGE_NIR image creation skipped; output directory already exists.


In [9]:
# DRG combination (NDVI, red, green)
ending = "RED_GREEN_NDVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_GREEN_NDVI image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NDVI" / img_name_ndvi
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_GREEN_NDVI image creation skipped; output directory already exists.")

RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.


RED_GREEN_NDVI image creation: 100%|██████████| 452/452 [00:22<00:00, 20.04it/s]


RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.


RED_GREEN_NDVI image creation: 100%|██████████| 235/235 [00:11<00:00, 20.14it/s]


RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.


RED_GREEN_NDVI image creation: 100%|██████████| 136/136 [00:06<00:00, 19.57it/s]

RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.
RED_GREEN_NDVI image creation skipped; output directory already exists.


In [10]:
# DO nen with zeroed blue channel
ending = "NIR_RED_NO_BLUE"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_RED_NO_BLUE image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ngrdi.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "nir" / img_name_nir,
                    band2=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ndvi,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, True]
            )

    else:
        print("NIR_RED_NO_BLUE image creation skipped; output directory already exists.")

NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.


NIR_RED_NO_BLUE image creation: 100%|██████████| 452/452 [00:37<00:00, 12.06it/s]


NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.


NIR_RED_NO_BLUE image creation: 100%|██████████| 235/235 [00:12<00:00, 18.19it/s]


NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.


NIR_RED_NO_BLUE image creation: 100%|██████████| 136/136 [00:06<00:00, 21.31it/s]

NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.
NIR_RED_NO_BLUE image creation skipped; output directory already exists.


In [11]:
# NIR_RED_NGRDI combination (nir, red, ngrdi)
ending = "NIR_RED_NGRDI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_RED_NGRDI image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_ngrdi = img_name.replace(".tif", "_ngrdi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "nir" / img_name_nir,
                    band2=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ngrdi,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("NIR_RED_NGRDI image creation skipped; output directory already exists.")

NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.


NIR_RED_NGRDI image creation: 100%|██████████| 452/452 [00:42<00:00, 10.70it/s]


NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.


NIR_RED_NGRDI image creation: 100%|██████████| 235/235 [00:12<00:00, 18.99it/s]


NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.


NIR_RED_NGRDI image creation: 100%|██████████| 136/136 [00:06<00:00, 20.25it/s]

NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.
NIR_RED_NGRDI image creation skipped; output directory already exists.


In [12]:
# DO nen with zeroed blue channel
ending = "NIR_GREEN_NO_BLUE"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_GREEN_NO_BLUE image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ngrdi.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ndvi,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band1=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, True]
            )

    else:
        print("NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.")

NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.


NIR_GREEN_NO_BLUE image creation: 100%|██████████| 452/452 [00:23<00:00, 19.29it/s]


NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.


NIR_GREEN_NO_BLUE image creation: 100%|██████████| 235/235 [00:11<00:00, 19.90it/s]


NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.


NIR_GREEN_NO_BLUE image creation: 100%|██████████| 136/136 [00:06<00:00, 20.41it/s]

NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.
NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.


In [13]:
# DO nen with zeroed blue channel
ending = "NGRDI_NDVI_RVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)
    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NGRDI_NDVI_RVI image creation"):
            if "xml" in img_name:
                continue
            img_name_ngrdi = img_name.replace(".tif", "_ngrdi.png")
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            img_name_rvi = img_name.replace(".tif", "_rvi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ngrdi,
                    band2=input_dir / "image_tiles_indeces_utf8" / "NDVI"  / img_name_ndvi,
                    band1=input_dir / "image_tiles_indeces_utf8" / "RVI"  / img_name_rvi,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, False]
            )

    else:
        print("NGRDI_NDVI_RVI image creation skipped; output directory already exists.")

NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.


NGRDI_NDVI_RVI image creation: 100%|██████████| 452/452 [00:30<00:00, 14.59it/s]


NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.


NGRDI_NDVI_RVI image creation: 100%|██████████| 235/235 [00:10<00:00, 22.14it/s]


NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.


NGRDI_NDVI_RVI image creation: 100%|██████████| 136/136 [00:05<00:00, 23.55it/s]

NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.
NGRDI_NDVI_RVI image creation skipped; output directory already exists.


In [18]:
# DO nen with zeroed blue channel
ending = "EXGR_GREEN_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists():
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="EXGR_GREEN_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_exgr = img_name.replace(".tif", "_exgr.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "image_tiles_indeces_utf8" / "EXGR" / img_name_exgr,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, False]
            )

    else:
        print("EXGR_GREEN_NIR image creation skipped; output directory already exists.")

EXGR_GREEN_NIR image creation:   0%|          | 0/473 [00:00<?, ?it/s]


RasterioIOError: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal/processed_output/image_tiles_indeces_utf8/EXGR/Bjornkjaervej_TestFlight_2_bigger_tile_10_10_exgr.png: No such file or directory